In [3]:
!pip install transformers datasets pytorch-crf scikit-learn

In [4]:
"""
NER 模型: BERT + BiLSTM + MEGA (双分支加权 Loss)
平台: Kaggle (GPU T4 / P100)
数据集: PassbyGrocer/msra-ner

修复说明:
  新版 transformers (>=4.40) 的 PreTrainedModel.__init__ 会调用
  sys.modules[cls.__module__].__file__，在 Kaggle notebook 的
  __main__ 环境中该属性不存在，导致 AttributeError。
  解决方案：主模型直接继承 nn.Module，完全绕开基类检查。
"""

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from datasets import load_dataset
from transformers import (
    BertModel, BertTokenizerFast, BertConfig,
    Trainer, TrainingArguments, TrainerCallback, logging,
    DataCollatorForTokenClassification,
)
from sklearn.metrics import f1_score

logging.set_verbosity_error()


# ─────────────────────────────────────────────────────────────────────────────
# 0.  Trainer 兼容的输出容器
#     Trainer 要求模型返回对象的第一个字段是 loss（或 dict 含 "loss" key）。
#     直接返回 dict 即可，Trainer 能识别 {"loss": ..., "logits": ...}。
# ─────────────────────────────────────────────────────────────────────────────

# forward 直接返回普通 dict，Trainer 原生支持 {"loss": ..., "logits": ...}
# ─────────────────────────────────────────────────────────────────────────────
# 1.  MEGA 核心组件
# ─────────────────────────────────────────────────────────────────────────────

class EMALayer(nn.Module):
    """
    Multi-dimensional Exponential Moving Average (EMA)
    h_t = α * h_{t-1} + (1-α) * x_t
    输入/输出: (B, L, d_model)
    """
    def __init__(self, d_model: int, d_state: int = 16):
        super().__init__()
        self.d_state = d_state
        self.alpha = nn.Parameter(torch.randn(d_model, d_state) * 0.02)
        self.delta = nn.Parameter(torch.randn(d_model, d_state) * 0.02)
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        alpha = torch.sigmoid(self.alpha)          # (D, S)
        delta = torch.sigmoid(self.delta)          # (D, S)

        h = x.new_zeros(B, D, self.d_state)
        outputs = []
        for t in range(L):
            xt = x[:, t, :]                        # (B, D)
            h = alpha * h + (1 - alpha) * xt.unsqueeze(-1)
            yt = (h * delta).sum(-1)               # (B, D)
            outputs.append(yt)

        out = torch.stack(outputs, dim=1)          # (B, L, D)
        return out * self.gamma + self.beta


class MEGABlock(nn.Module):
    """
    MEGA: Moving Average Equipped Gated Attention
    EMA → Gated Single-Head Attention → FFN → 残差
    """
    def __init__(self, d_model: int, d_attn: int = 256,
                 d_ffn: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ema   = EMALayer(d_model)
        self.q_proj = nn.Linear(d_model, d_attn, bias=False)
        self.k_proj = nn.Linear(d_model, d_attn, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.gate   = nn.Linear(d_model, d_model, bias=True)
        self.out    = nn.Linear(d_model, d_model, bias=False)
        self.scale  = math.sqrt(d_attn)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ffn), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ffn, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        residual = x
        x = self.norm1(x)
        ema_out = self.ema(x)

        Q = self.q_proj(ema_out)
        K = self.k_proj(ema_out)
        V = self.v_proj(ema_out)
        attn = torch.bmm(Q, K.transpose(1, 2)) / self.scale

        if attention_mask is not None:
            mask = (1 - attention_mask).bool().unsqueeze(1)
            attn = attn.masked_fill(mask, float('-inf'))

        attn = self.drop(F.softmax(attn, dim=-1))
        ctx  = torch.bmm(attn, V) * torch.sigmoid(self.gate(ema_out))
        x    = self.drop(self.out(ctx)) + residual
        x    = x + self.drop(self.ffn(self.norm2(x)))
        return x


# ─────────────────────────────────────────────────────────────────────────────
# 2.  主模型 —— 继承纯 nn.Module，完全绕开 transformers 基类的 __file__ 检查
# ─────────────────────────────────────────────────────────────────────────────

class BertBiLSTMMEGAForNER(nn.Module):
    """
    双分支 NER 模型:
        BERT → dropout
            ├─ BiLSTM → Linear → logits₁ → Loss₁
            └─ MEGA   → Linear → logits₂ → Loss₂
        总损失 = alpha_loss·Loss₁ + (1-alpha_loss)·Loss₂
    推理: logits = (logits₁ + logits₂) / 2
    """
    def __init__(self, bert_model: BertModel,
                 num_labels: int, alpha_loss: float = 0.5):
        super().__init__()
        self.num_labels = num_labels
        self.alpha_loss = alpha_loss

        self.bert    = bert_model
        H            = bert_model.config.hidden_size   # 768
        self.dropout = nn.Dropout(0.3)

        # 分支 1: BiLSTM
        self.bilstm = nn.LSTM(
            input_size=H, hidden_size=H // 2,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.3,
        )
        self.classifier_lstm = nn.Linear(H, num_labels)

        # 分支 2: MEGA
        self.mega            = MEGABlock(d_model=H, d_attn=256,
                                         d_ffn=H * 4, dropout=0.1)
        self.classifier_mega = nn.Linear(H, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                token_type_ids=None, labels=None, **kwargs):

        seq = self.dropout(
            self.bert(input_ids, attention_mask=attention_mask,
                      token_type_ids=token_type_ids).last_hidden_state
        )                                              # (B, L, H)

        # 分支 1
        lstm_out, _ = self.bilstm(seq)
        logits1 = self.classifier_lstm(lstm_out)       # (B, L, C)

        # 分支 2
        mega_out = self.mega(seq, attention_mask=attention_mask)
        logits2  = self.classifier_mega(mega_out)      # (B, L, C)

        # 推理: 平均
        logits = (logits1 + logits2) / 2.0

        loss = None
        if labels is not None:
            ce    = nn.CrossEntropyLoss()
            loss1 = ce(logits1.view(-1, self.num_labels), labels.view(-1))
            loss2 = ce(logits2.view(-1, self.num_labels), labels.view(-1))
            loss  = self.alpha_loss * loss1 + (1 - self.alpha_loss) * loss2

        return {"loss": loss, "logits": logits}


# ─────────────────────────────────────────────────────────────────────────────
# 3.  数据预处理与评估
# ─────────────────────────────────────────────────────────────────────────────

def tokenize_and_align_labels(examples, tokenizer, max_length: int = 128):
    tokenized = tokenizer(
        examples["tokens"], truncation=True,
        is_split_into_words=True, max_length=max_length,
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids  = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            prev_word = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized


def compute_metrics(eval_preds, label_list):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    true_labels = [[label_list[l] for p, l in zip(pr, la) if l != -100]
                   for pr, la in zip(preds, labels)]
    true_preds  = [[label_list[p] for p, l in zip(pr, la) if l != -100]
                   for pr, la in zip(preds, labels)]
    flat_l = [i for row in true_labels for i in row]
    flat_p = [i for row in true_preds  for i in row]
    return {"f1": f1_score(flat_l, flat_p, average="weighted")}


# ─────────────────────────────────────────────────────────────────────────────
# 4.  进度回调
# ─────────────────────────────────────────────────────────────────────────────

class KaggleProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[Step {state.global_step:5d}] "
                  f"train_loss={logs['loss']:.4f}  "
                  f"lr={logs.get('learning_rate', 0):.2e}")
        if "eval_loss" in logs:
            print(f"[Epoch {logs.get('epoch', 0):.1f}] "
                  f"eval_loss={logs['eval_loss']:.4f}  "
                  f"F1={logs.get('eval_f1', 0):.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# 5.  主程序
# ─────────────────────────────────────────────────────────────────────────────

def main():
    MODEL_PATH = (
        "/kaggle/input/bert-base-chinese"
        if os.path.exists("/kaggle/input/bert-base-chinese")
        else "bert-base-chinese"
    )
    OUTPUT_DIR  = "/kaggle/working/results"
    LOGGING_DIR = "/kaggle/working/logs"

    ALPHA_LOSS   = 0.5
    EPOCHS       = 3
    BATCH_SIZE   = 32
    LR           = 3e-5
    MAX_LENGTH   = 128
    LOGGING_STEP = 20

    print(f"GPU 可用: {torch.cuda.is_available()}")
    print(f"模型路径: {MODEL_PATH}")
    print(f"Loss 权重 -> BiLSTM: {ALPHA_LOSS:.2f} | MEGA: {1 - ALPHA_LOSS:.2f}")

    # ── 数据 ────────────────────────────────────────────────────────
    print("\n加载数据集...")
    dataset    = load_dataset("PassbyGrocer/msra-ner")
    label_list = dataset["train"].features["ner_tags"].feature.names
    print(f"标签 ({len(label_list)}): {label_list}")

    tokenizer    = BertTokenizerFast.from_pretrained(MODEL_PATH)
    tokenized_ds = dataset.map(
        lambda x: tokenize_and_align_labels(x, tokenizer, MAX_LENGTH),
        batched=True, remove_columns=dataset["train"].column_names,
        desc="Tokenizing",
    )

    # ── 模型 ─────────────────────────────────────────────────────────
    # 用 BertModel.from_pretrained 加载预训练权重（BertModel 本身继承
    # PreTrainedModel，但它在 transformers 包内有 __file__，不会触发报错）
    print("\n初始化模型...")
    bert_model = BertModel.from_pretrained(MODEL_PATH)
    model      = BertBiLSTMMEGAForNER(bert_model,
                                       num_labels=len(label_list),
                                       alpha_loss=ALPHA_LOSS)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"可训练参数量: {n_params / 1e6:.1f} M")

    # ── 训练参数 ─────────────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        learning_rate=LR,
        warmup_ratio=0.1,
        weight_decay=0.01,
        logging_steps=LOGGING_STEP,
        eval_strategy="epoch",
        save_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none",
        logging_dir=LOGGING_DIR,
        disable_tqdm=False,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        dataloader_num_workers=2,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=lambda x: compute_metrics(x, label_list),
        callbacks=[KaggleProgressCallback()],
    )

    print("\n开始训练...")
    trainer.train()

    # ── 保存 ─────────────────────────────────────────────────────────
    save_dir = "/kaggle/working/final_model"
    os.makedirs(save_dir, exist_ok=True)
    # 保存完整模型权重
    torch.save(model.state_dict(), os.path.join(save_dir, "model.pt"))
    # 保存 BERT config 和 tokenizer 供推理还原
    model.bert.config.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    print(f"\n训练完成！模型已保存至 {save_dir}")

    print("\n最终评估:")
    results = trainer.evaluate()
    for k, v in results.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


if __name__ == "__main__":
    main()

GPU 可用: True
模型路径: bert-base-chinese
Loss 权重 -> BiLSTM: 0.50 | MEGA: 0.50

加载数据集...
标签 (7): ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER']

初始化模型...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

可训练参数量: 116.3 M

开始训练...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.065878,0.052655,0.992577
2,0.029549,0.051875,0.993081
3,0.014171,0.055169,0.993794


...

[Step   740] train_loss=0.0352  lr=2.20e-05
[Step   760] train_loss=0.0369  lr=2.17e-05
[Step   780] train_loss=0.0366  lr=2.14e-05
[Step   800] train_loss=0.0395  lr=2.11e-05
[Step   820] train_loss=0.0385  lr=2.08e-05
[Step   840] train_loss=0.0362  lr=2.05e-05
[Step   860] train_loss=0.0301  lr=2.02e-05
[Step   880] train_loss=0.0357  lr=1.99e-05
[Step   900] train_loss=0.0286  lr=1.96e-05
[Step   920] train_loss=0.0276  lr=1.93e-05
[Step   940] train_loss=0.0283  lr=1.89e-05
[Step   960] train_loss=0.0373  lr=1.86e-05
[Step   980] train_loss=0.0310  lr=1.83e-05
[Step  1000] train_loss=0.0237  lr=1.80e-05
[Step  1020] train_loss=0.0336  lr=1.77e-05
[Step  1040] train_loss=0.0364  lr=1.74e-05
[Step  1060] train_loss=0.0247  lr=1.71e-05
[Step  1080] train_loss=0.0342  lr=1.68e-05
[Step  1100] train_loss=0.0319  lr=1.65e-05
[Step  1120] train_loss=0.0278  lr=1.62e-05
[Step  1140] train_loss=0.0326  lr=1.59e-05
[Step  1160] train_loss=0.0284  lr=1.56e-05
[Step  1180] train_loss=0.0243  

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[Step  1460] train_loss=0.0278  lr=1.10e-05
[Step  1480] train_loss=0.0160  lr=1.07e-05
[Step  1500] train_loss=0.0201  lr=1.04e-05
[Step  1520] train_loss=0.0159  lr=1.01e-05
[Step  1540] train_loss=0.0148  lr=9.75e-06
[Step  1560] train_loss=0.0176  lr=9.44e-06
[Step  1580] train_loss=0.0167  lr=9.14e-06
[Step  1600] train_loss=0.0158  lr=8.83e-06
[Step  1620] train_loss=0.0131  lr=8.52e-06
[Step  1640] train_loss=0.0184  lr=8.22e-06
[Step  1660] train_loss=0.0139  lr=7.91e-06
[Step  1680] train_loss=0.0142  lr=7.60e-06
[Step  1700] train_loss=0.0151  lr=7.30e-06
[Step  1720] train_loss=0.0131  lr=6.99e-06
[Step  1740] train_loss=0.0150  lr=6.68e-06
[Step  1760] train_loss=0.0156  lr=6.38e-06
[Step  1780] train_loss=0.0147  lr=6.07e-06
[Step  1800] train_loss=0.0158  lr=5.76e-06
[Step  1820] train_loss=0.0159  lr=5.46e-06
[Step  1840] train_loss=0.0123  lr=5.15e-06
[Step  1860] train_loss=0.0123  lr=4.84e-06
[Step  1880] train_loss=0.0126  lr=4.54e-06
[Step  1900] train_loss=0.0157  

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[Epoch 3.0] eval_loss=0.0552  F1=0.9938
  eval_loss: 0.0552
  eval_f1: 0.9938
  eval_runtime: 22.0609
  eval_samples_per_second: 197.8610
  eval_steps_per_second: 1.5870
  epoch: 3.0000
